# Exercises Week 4 with Solutions

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import random

from nltk.tokenize import word_tokenize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def TODO(todo: str = "Fill the blank"):
    raise ValueError(todo)
    
# Random Seed at file level
random_seed = 42

np.random.seed(random_seed)
random.seed(random_seed)

# Load the data into Python
texts, labels = [], []
with open("../data/lang_detection.txt", "r", encoding="utf-8") as f:
    for line in f:
        sentence, label = line.strip().split("\t")
        texts.append(sentence)
        labels.append(label)

y = np.array(labels)

In the lecture of week 4 we trained a very simple language detection model based on ASCII representations of characters (```simple_text_representation.ipynb```) as well as a Term-Document matrix (```tf-idf approach.ipynb```). 

While the first approach builds features on the character-level of text, the latter uses entire words to construct the feature matrix ```X```. 

### Exercise 4.1: Embeddings and Tokenization

Load the lang_detection.txt data into this notebook and create

1.) a vocabulary consisting of all the **characters** that appear in the dataset

2.) a vocabulary consisting of all the **words** that appear in the dataset

For both vocabularies, assign each item an ID and create an embedding matrix (```nn.Embedding```) for both vocabularies with an embedding dimension of 32. Take an arbitrary text from the dataset and encode it as a sequence of embeddings. 


In [ ]:
# Character vocabulary
all_texts_string = " ".join(texts)
character_vocab = TODO("Convert the text collection to a vocabulary of characters")

# assign each character an ID
char_to_id = TODO("assign each character an ID. Use a dictionary comprehension for this")
print(len(char_to_id))

In [ ]:
def character_encoder(text: str):
    "Takes some arbirary text and encodes it on character-level using the char_to_id dictionary"
    return [char_to_id[char] for char in text]

In [ ]:
# Word vocabulary
all_words = []
for text in texts:
    all_words.extend(TODO("use the word tokenizer from nltk and add the words to the list"))
word_vocab = sorted(list(set(all_words)))

# assign each word an ID
word_to_id = TODO("assign each word an ID. Use a dictionary comprehension for this")
print(len(word_to_id))

In [ ]:
def word_encoder(text: str):
    "Takes some arbirary text and encodes it on word-level using the word_to_id map"
    words = [w.lower() for w in word_tokenize(text) if w.isalpha()]
    return [word_to_id[word] for word in words]

In [ ]:
char_emb_table = nn.Embedding(TODO("Pass the correct parameter values for num_embeddings and embedding_dim"))
word_emb_table = nn.Embedding(TODO("Pass the correct parameter values for num_embeddings and embedding_dim"))

In [ ]:
def get_embed_seq(text: str, emb_table: nn.Embedding, encoder_fn):
    "Takes some arbirary text an embedding table and an encoder function to create a sequence of embeddings for the text"
    text_ids = TODO("which function to use here?")
    id_tensor = torch.tensor(TODO(), dtype=torch.long)  # ID tensors always require torch.long as data type
    embeddings = emb_table(id_tensor)
    return embeddings

In [ ]:
# apply the function for word- and character-level tokens on some example text
example_text = texts[np.random.choice(list(range(len(texts))))]
print("Example text: ", example_text)
# apply function for word and character-level tokens
word_emb_seq = get_embed_seq(example_text, word_emb_table, word_encoder)
print("Word embedding sequence dimensions: ", word_emb_seq.shape)

char_emb_seq = get_embed_seq(example_text, char_emb_table, character_encoder)
print("Character embedding sequence dimensions: ",char_emb_seq.shape)

### Questions: 
1. What dimensions does the embedding matrix have in each case?
2. What dimensions does the embedding sequence of the example text have?
3. What might be the Pros and Cons of either approach?

### Potential Answers: 
1. Character-level: [32,32] ; Word-level: [360, 32]
2. Character-level: [12,32] ; Word-level: [3, 32]
3. Pros/Cons:
| Tokenization Type   | Pros                                                                                                                    | Cons                                                                                                      |
| ------------------- | ----------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- |
| **Word-level**      | Easy to interpret (tokens are actual words)<br>Captures meaning at a natural level<br>Shorter sequences than characters | Large vocabulary size<br>Struggles with unknown or rare words<br>Language-dependent (needs preprocessing) |
| **Character-level** | Very small vocabulary<br>Handles any word (no unknown tokens)<br>Language-agnostic                                      | Very long sequences<br>Harder to learn meaningful patterns<br>Slower training and inference               |


----

### Exercise 4.2 N-Grams and Bag-of-Words

Character- and word-level tokenization both come with their problems. One approach that sits in between the two is that of using $n$-gram tokenization, where sequences of $n$ consecutive characters are treated as tokens. This allows models to capture short, meaningful pieces of words (like prefixes or suffixes) instead of just single characters, while still being more flexible than full word-based approaches. Because each token now represents a small chunk rather than a single letter, the model has to process far fewer tokens overall, which makes learning faster and more efficient.

Use the function ```get_trigrams```below to generate a vocabulary of all possible 3-grams. Again, assign each item in the vocabulary an ID. Given this vocabulary, generate a 3-gram-document matrix (counting the appearance of the 3-grams in each document - like a term-document matrix but with 3-grams instead of words). Use this 3-gram-document matrix as input to a ```LogisticRegression``` model to predict language

In [ ]:
def get_trigrams(word):
    """This function generates the 3-grams for a given word"""
    extended_word = f"<{word}>"
    ngrams = []
    for i in range(len(extended_word) - 3 + 1):
        ngrams.append(extended_word[i:i+3])
    return ngrams

print(get_trigrams("Hello"))

In [ ]:
all_ngrams = []
for word in set(word_vocab):
    TODO("Fill the vocabulary")

ngram_vocab = sorted(list(set(all_ngrams)))
ngram_to_id = TODO("Define the mapping from 3-gram to IDs")
N_GRAM_V = len(ngram_vocab)

In [ ]:
def text_to_trigram_freq(text):
    freq_vec = np.zeros((N_GRAM_V))  # initialize the frequence vector
    words  = TODO("Split the text into words first")
    for word in words:
        TODO("Determine the 3-grams of the word and increment the freq_vec accordingly")
    return freq_vec

In [ ]:
# Here we generate the feature matrix X
X = np.zeros((len(texts), N_GRAM_V))
for i, text in enumerate(texts):
    X[i,:] = text_to_trigram_freq(text)

In [ ]:
# Split and Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)

TODO("Fit the LogisticRegression model from sklearn and generate y_pred (the predictions on the test data)")

In [ ]:
f"The accurary for this model is {float((y_pred == y_test).sum() / len(y_test) * 100)}%"

### Questions: 
1. How does the accuracy of this model compare to the other approaches covered in the lecture?
2. How do you explain this change in accuracy?

### Answers:

1. The $n$-gram model achieves higher accuracy than both character-level and word-level tokenization approaches discussed in the lecture.
2. This improvement comes from a better balance between the two extremes: unlike character-level models, $n$-grams capture short, meaningful patterns within words (e.g., prefixes, suffixes), and unlike word-level models, they avoid issues with rare or unseen words. This leads to more informative tokens and shorter sequences, which makes it easier for the model to learn useful patterns.